In [0]:
"""
id: clinical_trials_src
template: source
templateVersion: 2.0.0
name: clinical_trials
position:
  x: 0
  y: 10
description:
  text: Read data from a JSON file with headers and inferred schema.
  hash: 9f743faa
previewCodeHash: b31b6f9aee0d90fa
previewMode: "1000"
config:
  table_source:
    tableName: pharma.bronze.clinical_trials
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "pharma.bronze.clinical_trials"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["clinical_trials_src.data"] = out["data"]

In [0]:
"""
id: drug_inventory_src
template: source
templateVersion: 2.0.0
name: drug_inventory
position:
  x: 0
  y: 630
description:
  text: Load all data from the drug inventory table.
  hash: 31dac1b0
previewCodeHash: 73964fe007c3f15b
previewMode: "1000"
config:
  table_source:
    tableName: pharma.bronze.drug_inventory
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "pharma.bronze.drug_inventory"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["drug_inventory_src.data"] = out["data"]

In [0]:
"""
id: pharma_github_adverse_events
template: python
templateVersion: 1.0.0
name: pharma_github_adverse_events
position:
  x: 0
  y: 320
description:
  text: Fetch data from a web source and add columns for the source URL and fetch time.
  hash: 7905df24
previewCodeHash: 3c30d77335b94526
previewMode: "1000"
config:
  code: |-
    import requests
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import current_timestamp, lit
    GITHUB_USER = "mahendraboopathitm"
    REPO        = "lakeflow_designer"
    BRANCH      = "main"
    BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}"

    spark = SparkSession.builder.getOrCreate()
    url  = f"{BASE_URL}/adverse_events.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    df = spark.createDataFrame(data)
    df = (df
        .withColumn("_api_source", lit(url))
        .withColumn("_fetched_at", current_timestamp())
    )
    result = df
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    import requests
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import current_timestamp, lit
    GITHUB_USER = "mahendraboopathitm"
    REPO        = "lakeflow_designer"
    BRANCH      = "main"
    BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}"

    spark = SparkSession.builder.getOrCreate()
    url  = f"{BASE_URL}/adverse_events.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    df = spark.createDataFrame(data)
    df = (df
        .withColumn("_api_source", lit(url))
        .withColumn("_fetched_at", current_timestamp())
    )
    result = df

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["pharma_github_adverse_events.result"] = out["result"]

In [0]:
"""
id: pharma_github_patient_demographics
template: python
templateVersion: 1.0.0
name: pharma_github_patient_demographics
position:
  x: 0
  y: 940
description:
  text: Load patient demographic data from a GitHub repository and add source and fetch time information.
  hash: 6a9dcecc
previewCodeHash: f4ac9d2434e4c2e3
previewMode: "1000"
config:
  code: |
    import requests
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import current_timestamp, lit
    GITHUB_USER = "mahendraboopathitm"
    REPO        = "lakeflow_designer"
    BRANCH      = "main"
    BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}"

    spark = SparkSession.builder.getOrCreate()
    url  = f"{BASE_URL}/patient_demographics%20(1).json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    df = spark.createDataFrame(data)
    df = (df
        .withColumn("_api_source", lit(url))
        .withColumn("_fetched_at", current_timestamp())
    )
    result = df
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    import requests
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import current_timestamp, lit
    GITHUB_USER = "mahendraboopathitm"
    REPO        = "lakeflow_designer"
    BRANCH      = "main"
    BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/{BRANCH}"

    spark = SparkSession.builder.getOrCreate()
    url  = f"{BASE_URL}/patient_demographics%20(1).json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    df = spark.createDataFrame(data)
    df = (df
        .withColumn("_api_source", lit(url))
        .withColumn("_fetched_at", current_timestamp())
    )
    result = df

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["pharma_github_patient_demographics.result"] = out["result"]

In [0]:
"""
id: clinical_trials_bronze
template: output
templateVersion: 1.0.0
name: pharma.bronze.clinical_trials_bronze
position:
  x: 300
  y: 0
description:
  text: Save data to specified table, replacing existing content.
  hash: eb55bc66
previewMode: "1000"
config:
  catalog: pharma
  schema: bronze
  table_name: clinical_trials_bronze
input:
  - node: clinical_trials_src
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "pharma",
    "schema": "bronze",
    "table_name": "clinical_trials_bronze"
}
inputs = {
    "data": ctx["clinical_trials_src.data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: clinical_trials_silver
template: transform
templateVersion: 1.0.0
name: clinical_trials_silver
position:
  x: 300
  y: 465
description:
  text: Calculate trial completion percentage and group trial statuses while keeping patient counts.
  hash: 97db2765
previewCodeHash: 8232215ac40af1f8
previewMode: "1000"
config:
  expressions:
    - completed_patient_count
    - patient_count
    - (completed_patient_count / patient_count) * 100 AS `trial_completion_pct`
    - CASE WHEN status = 'Completed' THEN 'Completed' WHEN status = 'Terminated' THEN 'Terminated' WHEN status = 'Active' THEN 'Active' WHEN status = 'Suspended' THEN 'Suspended' ELSE status END AS `trial_status_group`
    - CASE WHEN trial_completion_pct >= 90 THEN 'Excellent' WHEN trial_completion_pct >= 70 THEN 'Good' WHEN trial_completion_pct >= 50 THEN 'Moderate' ELSE 'Low' END AS `trial_health_score`
    - trial_id
input:
  - node: clinical_trials_src
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "completed_patient_count",
        "patient_count",
        "(completed_patient_count / patient_count) * 100 AS `trial_completion_pct`",
        "CASE WHEN status = 'Completed' THEN 'Completed' WHEN status = 'Terminated' THEN 'Terminated' WHEN status = 'Active' THEN 'Active' WHEN status = 'Suspended' THEN 'Suspended' ELSE status END AS `trial_status_group`",
        "CASE WHEN trial_completion_pct >= 90 THEN 'Excellent' WHEN trial_completion_pct >= 70 THEN 'Good' WHEN trial_completion_pct >= 50 THEN 'Moderate' ELSE 'Low' END AS `trial_health_score`",
        "trial_id"
    ]
}
inputs = {
    "data": ctx["clinical_trials_src.data"]
}
out = run(config, inputs, spark)
ctx["clinical_trials_silver.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: drug_inventory_bronze
template: output
templateVersion: 1.0.0
name: drug_inventory_bronze
position:
  x: 300
  y: 620
description:
  text: Save data to a specified table, replacing existing content.
  hash: 36ee2fab
previewMode: "1000"
config:
  catalog: pharma
  schema: bronze
  table_name: drug_inventory_bronze
input:
  - node: drug_inventory_src
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "pharma",
    "schema": "bronze",
    "table_name": "drug_inventory_bronze"
}
inputs = {
    "data": ctx["drug_inventory_src.data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: drug_inventory_silver
template: transform
templateVersion: 1.0.0
name: drug_inventory_silver
position:
  x: 300
  y: 775
description:
  text: Add columns to show inventory status and whether items are expired.
  hash: 8e14f1c2
previewCodeHash: 9cbf1babc876fc80
previewMode: "1000"
config:
  expressions:
    - CASE WHEN quantity_mg < reorder_level_mg THEN 'Low Stock' ELSE 'Healthy Stock' END AS `inventory_status`
    - CASE WHEN expiry_date < CURRENT_DATE() THEN true ELSE false END AS `expired_flag`
    - CASE WHEN quantity_mg < 100 THEN 'Critical' WHEN quantity_mg < 500 THEN 'Warning' ELSE 'Normal' END AS `inventory_risk_segment`
    - approved_by
    - drug_id
input:
  - node: drug_inventory_src
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "CASE WHEN quantity_mg < reorder_level_mg THEN 'Low Stock' ELSE 'Healthy Stock' END AS `inventory_status`",
        "CASE WHEN expiry_date < CURRENT_DATE() THEN true ELSE false END AS `expired_flag`",
        "CASE WHEN quantity_mg < 100 THEN 'Critical' WHEN quantity_mg < 500 THEN 'Warning' ELSE 'Normal' END AS `inventory_risk_segment`",
        "approved_by",
        "drug_id"
    ]
}
inputs = {
    "data": ctx["drug_inventory_src.data"]
}
out = run(config, inputs, spark)
ctx["drug_inventory_silver.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: adverse_events_bronze
template: output
templateVersion: 1.0.0
name: adverse_events_bronze
position:
  x: 300
  y: 310
description:
  text: Overwrite existing data in the specified table.
  hash: 0792b9ff
previewMode: "1000"
config:
  catalog: pharma
  schema: bronze
  table_name: adverse_events_bronze
input:
  - node: pharma_github_adverse_events
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "pharma",
    "schema": "bronze",
    "table_name": "adverse_events_bronze"
}
inputs = {
    "data": ctx["pharma_github_adverse_events.result"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: adverse_events_silver
template: transform
templateVersion: 1.0.0
name: adverse_events_silver
position:
  x: 300
  y: 1085
description:
  text: Classify risk as high or normal based on severity or adverse event, and extract event year.
  hash: "0e352081"
previewCodeHash: 39779d698df7f464
previewMode: "1000"
config:
  expressions:
    - patient_id
    - CASE WHEN severity = 'Severe' OR sae = true THEN 'High Risk' ELSE 'Normal Risk' END AS `high_risk_flag`
    - YEAR(reported_date) AS `event_year`
    - CASE WHEN severity = 'Severe' THEN 5 WHEN severity = 'Moderate' THEN 3 WHEN severity = 'Mild' THEN 1 ELSE 0 END AS `severity_score`
input:
  - node: pharma_github_adverse_events
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "patient_id",
        "CASE WHEN severity = 'Severe' OR sae = true THEN 'High Risk' ELSE 'Normal Risk' END AS `high_risk_flag`",
        "YEAR(reported_date) AS `event_year`",
        "CASE WHEN severity = 'Severe' THEN 5 WHEN severity = 'Moderate' THEN 3 WHEN severity = 'Mild' THEN 1 ELSE 0 END AS `severity_score`"
    ]
}
inputs = {
    "data": ctx["pharma_github_adverse_events.result"]
}
out = run(config, inputs, spark)
ctx["adverse_events_silver.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: patient_demographics_bronze
template: output
templateVersion: 1.0.0
name: patient_demographics_bronze
position:
  x: 300
  y: 930
description:
  text: Save the data into the specified table, replacing anything already there.
  hash: 528f0447
previewMode: "1000"
config:
  catalog: pharma
  schema: bronze
  table_name: patient_demographics_bronze
input:
  - node: pharma_github_patient_demographics
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "pharma",
    "schema": "bronze",
    "table_name": "patient_demographics_bronze"
}
inputs = {
    "data": ctx["pharma_github_patient_demographics.result"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: patient_demographics_silver
template: transform
templateVersion: 1.0.0
name: patient_demographics_silver
position:
  x: 300
  y: 155
description:
  text: Create new columns to standardize gender, categorize age into groups, and classify BMI.
  hash: 71705d16
previewCodeHash: 152a01f95ec8bb79
previewMode: "1000"
config:
  expressions:
    - patient_id
    - CASE WHEN LOWER(gender) LIKE '%male%' THEN 'Male' WHEN LOWER(gender) LIKE '%female%' THEN 'Female' ELSE 'Other' END AS `gender_standard`
    - CASE WHEN age BETWEEN 0 AND 18 THEN '0-18' WHEN age BETWEEN 19 AND 35 THEN '19-35' WHEN age BETWEEN 36 AND 60 THEN '36-60' ELSE '60+' END AS `age_group`
    - CASE WHEN bmi < 18.5 THEN 'Underweight' WHEN bmi >= 18.5 AND bmi < 25 THEN 'Normal' WHEN bmi >= 25 AND bmi < 30 THEN 'Overweight' ELSE 'Obese' END AS `bmi_category`
    - CASE WHEN smoker = true AND bmi >= 30 THEN 'Very High Risk' WHEN smoker = true AND bmi >= 25 THEN 'High Risk' WHEN smoker = true THEN 'Moderate Risk' WHEN bmi >= 30 THEN 'High Risk' ELSE 'Normal Risk' END AS `risk_segment`
    - drug_id
    - trial_id
input:
  - node: pharma_github_patient_demographics
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "patient_id",
        "CASE WHEN LOWER(gender) LIKE '%male%' THEN 'Male' WHEN LOWER(gender) LIKE '%female%' THEN 'Female' ELSE 'Other' END AS `gender_standard`",
        "CASE WHEN age BETWEEN 0 AND 18 THEN '0-18' WHEN age BETWEEN 19 AND 35 THEN '19-35' WHEN age BETWEEN 36 AND 60 THEN '36-60' ELSE '60+' END AS `age_group`",
        "CASE WHEN bmi < 18.5 THEN 'Underweight' WHEN bmi >= 18.5 AND bmi < 25 THEN 'Normal' WHEN bmi >= 25 AND bmi < 30 THEN 'Overweight' ELSE 'Obese' END AS `bmi_category`",
        "CASE WHEN smoker = true AND bmi >= 30 THEN 'Very High Risk' WHEN smoker = true AND bmi >= 25 THEN 'High Risk' WHEN smoker = true THEN 'Moderate Risk' WHEN bmi >= 30 THEN 'High Risk' ELSE 'Normal Risk' END AS `risk_segment`",
        "drug_id",
        "trial_id"
    ]
}
inputs = {
    "data": ctx["pharma_github_patient_demographics.result"]
}
out = run(config, inputs, spark)
ctx["patient_demographics_silver.transformed_data"] = out["transformed_data"]

In [0]:
"""
id: patient_adverse_events_join
template: join
templateVersion: 1.0.0
name: patient_adverse_events_join
position:
  x: 600
  y: 562.5
previewCodeHash: 9fa2c190f1571c9c
previewMode: "1000"
config:
  join_type: inner
  join_conditions: left.patient_id = right.patient_id
  expressions: []
input:
  - node: patient_demographics_silver
    input_port: left
    output_port: transformed_data
  - node: adverse_events_silver
    input_port: right
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "join_conditions": "left.patient_id = right.patient_id",
    "expressions": []
}
inputs = {
    "left": ctx["patient_demographics_silver.transformed_data"],
    "right": ctx["adverse_events_silver.transformed_data"]
}
out = run(config, inputs, spark)
ctx["patient_adverse_events_join.joined_data"] = out["joined_data"]

In [0]:
"""
id: patient_clinical_trials_join
template: join
templateVersion: 1.0.0
name: patient_clinical_trials_join
position:
  x: 600
  y: 950
previewCodeHash: 542a0e0625d025e5
previewMode: "1000"
config:
  join_type: inner
  join_conditions: left.trial_id = right.trial_id
  expressions: []
input:
  - node: patient_demographics_silver
    input_port: left
    output_port: transformed_data
  - node: clinical_trials_silver
    input_port: right
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "join_conditions": "left.trial_id = right.trial_id",
    "expressions": []
}
inputs = {
    "left": ctx["patient_demographics_silver.transformed_data"],
    "right": ctx["clinical_trials_silver.transformed_data"]
}
out = run(config, inputs, spark)
ctx["patient_clinical_trials_join.joined_data"] = out["joined_data"]

In [0]:
"""
id: patient_drug_inventory_join
template: join
templateVersion: 1.0.0
name: patient_drug_inventory_join
position:
  x: 600
  y: 1105
previewCodeHash: 66ecf83cf425912f
previewMode: "1000"
config:
  join_type: inner
  join_conditions: left.drug_id = right.drug_id
  expressions: []
input:
  - node: patient_demographics_silver
    input_port: left
    output_port: transformed_data
  - node: drug_inventory_silver
    input_port: right
    output_port: transformed_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "join_conditions": "left.drug_id = right.drug_id",
    "expressions": []
}
inputs = {
    "left": ctx["patient_demographics_silver.transformed_data"],
    "right": ctx["drug_inventory_silver.transformed_data"]
}
out = run(config, inputs, spark)
ctx["patient_drug_inventory_join.joined_data"] = out["joined_data"]

In [0]:
"""
id: gold_patient_safety_view
template: join
templateVersion: 1.0.0
name: Gold_Patient_Safety_View
position:
  x: 900
  y: 1105
description:
  text: Join patient clinical trials data with drug inventory data on patient ID using an inner join.
  hash: 054de795
previewCodeHash: 926ef7d6c6c5a6d6
previewMode: "1000"
config:
  join_type: inner
  join_conditions: left.patient_id = right.patient_id
  expressions: []
input:
  - node: patient_clinical_trials_join
    input_port: left
    output_port: joined_data
  - node: patient_drug_inventory_join
    input_port: right
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "join_conditions": "left.patient_id = right.patient_id",
    "expressions": []
}
inputs = {
    "left": ctx["patient_clinical_trials_join.joined_data"],
    "right": ctx["patient_drug_inventory_join.joined_data"]
}
out = run(config, inputs, spark)
ctx["gold_patient_safety_view.joined_data"] = out["joined_data"]